## data validation base dtype class classification

## import package

In [ ]:
import datetime
import pandas as pd
import numpy as np
import os
import sys
import pytz
from numpy.typing import ArrayLike, DTypeLike, NDArray
from typing import Sequence, List, Dict, Tuple, Set, Any, Union, Optional, Annotated, Callable, Literal, TypeVar
from loguru import logger
from dataclasses import dataclass, field
from pydantic import Field, BaseModel

# np.number, pd.Int64,..., int, float

## common config

In [ ]:
FILE_PATH: str = "/home/user/data-da-ds-de/prj1mrdp/data/raw/sales_data_sample.xlsx"
SHEET_NAME: str = "Sales_Data"

In [ ]:
DATE_FORMAT: str = "%Y-%m-%d"
DATETIME_FORMAT: str = "%Y-%m-%d HH:MM:SS"
today: str = datetime.datetime.now().strftime(DATE_FORMAT)
today

In [ ]:
logger.remove()

logger.add(
    sys.stdout,
    colorize=True,
    format="<level>[{level}]</level>[<green>{time:YYYY-MM-DD HH:mm:ss}</green>][<cyan>{name}:{function}:{line}</cyan>] <level>{message}</level>"
)

logger.add(
    f"/home/user/data-da-ds-de/data_validation/logs/{today}.log",
    colorize=False,
    format="[{level}][{time:YYYY-MM-DD HH:mm:ss}][{name}:{function}:{line}] {message}"
)

In [ ]:
INT_TYPE = (int, np.integer)
FLOAT_TYPE = (float, np.floating) + INT_TYPE

## common function

## IntValidation

In [ ]:
@dataclass
class IntValidation:
    __data: field(default_factory=lambda: pd.Series(dtype="object"))

    @property
    def data(self) -> pd.Series:
        return self.__data

    @data.setter
    def data(self, data: pd.Series = pd.Series()) -> None:
        self.__data = data

    @property
    def name(self) -> str:
        return self.__data.name

    @name.setter
    def name(self, name: str = "") -> None:
        self.__data.name = name

    @logger.catch
    def is_int(self) -> pd.Index:
        """Return index of value is int type."""
        sr: pd.Series = self.__data.dropna()
        return sr[sr.map(lambda _: isinstance(_, INT_TYPE))].index

    def __call__(self) -> pd.Index:
        return self.is_int()



In [ ]:
# test
df_int_validation: pd.Series = pd.Series([
    1,
    np.nan,
    "a"
    ], name="int_check")
df_int_validation

In [ ]:
df_int_validation_check: IntValidation = IntValidation(df_int_validation)
df_int_validation_check()

### numeric common

In [ ]:
@dataclass
class NumericValidation:
    # ---------- Data setup ----------
    __data: pd.Series = field(default_factory=lambda: pd.Series(dtype="object"))
    __is_int_check: bool = False
    __range: Tuple[int|float, int|float] = field(default_factory=lambda: (0, 1_000_000))
    __decimal_places: int = 2
    __result: Dict[str, pd.Index | None] = field(
        default_factory=lambda: {
            "not_int": None,
            "not_numeric": None,
            "not_in_range": None,
            "wrong_decimal_places": None
        }
    )

    # ---------- Getter/setter ----------
    @property
    def data(self) -> pd.Series:
        return self.__data

    @data.setter
    def data(self, data: pd.Series) -> None:
        self.__data = data

    @property
    def range(self) -> Tuple[int|float, int|float]:
        return self.__range

    @range.setter
    def range(self, range: Tuple[int|float, int|float]) -> None:
        self.__range = range

    @property
    def decimal_places(self) -> int:
        return self.__decimal_places

    @decimal_places.setter
    def decimal_places(self, decimal_places: int) -> None:
        self.__decimal_places = decimal_places

    @property
    def is_int_check(self) -> bool:
        return self.__is_int_check

    @is_int_check.setter
    def is_int_check(self, is_int_check: bool) -> None:
        self.__is_int_check = is_int_check

    # ---------- Validation function ----------
    # @logger.catch
    # def not_int(self) -> pd.Index:
    #     """Return index of value is not int type."""
    #     sr: pd.Series = self.__data.dropna()
    #     return sr[sr.map(lambda _: not isinstance(_, INT_TYPE))].index

    @logger.catch
    def not_numeric(self) -> pd.Index:
        """Return index of value is not numeric type."""
        sr: pd.Series = self.__data.dropna()
        return sr[pd.to_numeric(sr, errors="coerce").isna()].index

    @logger.catch
    def not_in_range(self) -> pd.Index:
        """Return index of value not in range."""
        low, high = sorted(self.__range)
        sr = pd.to_numeric(self.__data.dropna(), errors="coerce").dropna()
        return sr[(sr < low) | (sr > high)].index

    @staticmethod
    @logger.catch
    def split_decimal(_) -> bool:
        """"""
        decimal_str: str = str(_)
        if "." in decimal_str:
            return len(decimal_str.split(".")[1])
        else:
            return 0

    @logger.catch
    def wrong_decimal_places(self) -> pd.Index:
        """Return index of value has wrong decimal places."""
        sr = pd.to_numeric(self.__data.dropna(), errors="coerce").dropna()
        correct_index: pd.Index = sr[sr.map(lambda _: self.split_decimal(_) == self.__decimal_places)].index
        return sr.loc[~sr.index.isin(correct_index)].index

    def __call__(self):
        # Numeric/int/float type check
        if self.__is_int_check:
            self.__result["not_int"] = IntValidation(self.__data)()
            # self.not_int()
        else:
            self.__result["not_numeric"] = self.not_numeric()
            self.__result["wrong_decimal_places"] = self.wrong_decimal_places()

        self.__result["not_in_range"] = self.not_in_range()
        return self.__result

In [ ]:
# validation just return index of errors, add error id for main function or main class
# data to validation is not na, so before pass data, need preprocessing to just pass not empty/nan value

In [ ]:
test_df = pd.Series(
    [
        1,
        "a",
        1.1123
    ]
)
test_validation_df = NumericValidation(test_df, False, (2,0))

In [ ]:
test_validation_df()

## string

In [ ]:
# format, inner refer, outer refer, exist sub str,.. to customer validation => additional validation => class has function/attribute to add nore additional validation function
# mapping value => preprocessing => turn it into check in value list (here is mapped value list)
# so this is preprocessing before validation
@dataclass
class StringValidation:
    # ---------- Data setup ----------
    __data: pd.Series = field(default_factory=lambda: pd.Series(dtype="object"))
    __range: Tuple[int|float, int|float] = field(default_factory=lambda: (0, 1_000_000))
    __in_value_list: List | None = field(default_factory=lambda: None)
    __addtional_validation: List[Callable] | None = field(default_factory=lambda: None) # Need to think more
    __result: Dict[str, pd.Index | None] = field(
        default_factory=lambda: {
            "not_string": None,
            "not_in_length_range": None,
            "not_in_value_list": None,
        }
    )

    # ---------- Property ----------
    @property
    def data(self) -> pd.Series:
        return self.__data

    @data.setter
    def data(self, data: pd.Series = pd.Series()) -> None:
        self.__data = data

    @property
    def range(self) -> Tuple[int|float, int|float]:
        return self.__range

    @range.setter
    def range(self, range: Tuple[int|float, int|float]) -> None:
        self.__range = range

    @property
    def in_value_list(self) -> List | None:
        return self.__in_value_list

    @in_value_list.setter
    def in_value_list(self, in_value_list: List | None) -> None:
        self.__in_value_list = in_value_list

    # ---------- Function ----------
    @logger.catch
    def not_string(self) -> pd.Index:
        """Return index of value is not string type."""
        return self.__data[self.__data.dtypes != "string"].index

    @logger.catch
    def not_in_length_range(self) -> pd.Index:
        """Return index of value not in length range."""
        sr: pd.Series = self.__data
        sr = sr.loc[sr.index.isin(self.not_string())]
        low, high = sorted(self.__range)
        return sr[(sr.str.len() < low) | (sr.str.len() > high)].index

    @logger.catch
    def not_in_value_list(self) -> pd.Index:
        """Return index of value not in value list."""
        sr: pd.Series = self.__data
        sr = sr.loc[sr.index.isin(self.not_string())]
        return sr[~sr.isin(self.__in_value_list)].index

    def __call__(self):
        # String type check
        self.__result["not_string"] = self.not_string()
        # String length check
        self.__result["not_in_length_range"] = self.not_in_length_range()
        # String value check
        self.__result["not_in_value_list"] = self.not_in_value_list()
        return self.__result

## Datetime validation

In [ ]:
@dataclass
class DatetimeValidation:
    # ---------- Data setup ----------
    __data: pd.Series = field(default_factory=lambda: pd.Series(dtype="object"))
    __format: str = field(default_factory=lambda: "%Y-%m-%d")
    __range: Tuple[str, str] = field(default_factory=lambda: ("1900-01-01", "2100-01-01"))
    __result: Dict[str, pd.Index | None] = field(
        default_factory=lambda: {
            "not_datetime": None,
            "wrong_format": None,
            "not_in_range": None,
        }
    )

    # ---------- Property ----------
    @property
    def data(self) -> pd.Series:
        return self.__data

    @data.setter
    def data(self, data: pd.Series = pd.Series()) -> None:
        self.__data = data

    @property
    def range(self) -> Tuple[str, str]:
        return self.__range

    @range.setter
    def range(self, range: Tuple[str, str]) -> None:
        self.__range = range

    @property
    def format(self) -> str:
        return self.__format

    @format.setter
    def format(self, format: str) -> None:
        self.__format = format

    # ---------- Function ----------
    @logger.catch
    def not_datetime(self) -> pd.Index:
        """Return index of value is not datetime type."""
        return pd.to_datetime(self.__data, errors="coerce").isna().index

    @logger.catch
    def wrong_format(self) -> pd.Index:
        """Return index of value not in format."""
        sr: pd.Series = sr[~sr.index.isin(self.not_datetime())]
        return pd.to_datetime(sr, format=self.__format, errors="coerce").isna().index

    @logger.catch
    def not_in_range(self) -> pd.Index:
        """Return index of value not in range."""
        input_range: Tuple[pd.DatetimeTZDtype, pd.DatetimeTZDtype] = (pd.to_datetime(self.__range[0], format=self.__format), pd.to_datetime(self.__range[1], format=self.__format))
        low, high = sorted(input_range)
        return self.__data[(self.__data < low) | (self.__data > high)].index

    def __call__(self):
        # Datetime type check
        self.__result["not_datetime"] = self.not_datetime()
        # Datetime format check
        self.__result["wrong_format"] = self.wrong_format()
        # Datetime range check
        self.__result["not_in_range"] = self.not_in_range()
        return self.__result

## Boolean validation

In [ ]:
# preprocessing : mapping bool value
@dataclass
class BoolValidation:
    # ---------- Data setup ----------
    __data: pd.Series = field(default_factory=lambda: pd.Series(dtype="object"))
    __result: Dict[str, pd.Index | None] = field(
        default_factory=lambda: {
            "not_bool": None,
        }
    )

    # ---------- Property ----------
    @property
    def data(self) -> pd.Series:
        return self.__data

    @data.setter
    def data(self, data: pd.Series = pd.Series()) -> None:
        self.__data = data

    # ---------- Function ----------
    @logger.catch
    def not_bool(self) -> pd.Index:
        """Return index of value is not bool type."""
        return pd.to_numeric(self.__data, errors="coerce").isna().index

    def __call__(self):
        # Bool type check
        self.__result["not_bool"] = self.not_bool()
        return self.__result

## Additional validation


In [ ]:
@dataclass
class AdditionalValidation:
    # ---------- Data setup ----------
    __data: pd.Series = field(default_factory=lambda: pd.Series(dtype="object"))
    __dtype: str = field(default_factory=lambda: "object")
    __result: Dict[str, pd.Index | None] = field(
        default_factory=lambda: {
            "not_in_value_list": None,
        }
    )

    # ---------- Property ----------
    @property
    def data(self) -> pd.Series:
        return self.__data

    @data.setter
    def data(self, data: pd.Series = pd.Series()) -> None:
        self.__data = data

    @property
    def dtype(self) -> str:
        return self.__dtype

    @dtype.setter
    def dtype(self, dtype: str) -> None:
        self.__dtype = dtype

    # ---------- Function ----------
    @logger.catch
    def not_in_value_list(self) -> pd.Index:
        """Return index of value not in value list."""
        return self.__data[~self.__data.isin(self.value_list)].index

    def __call__(self):
        # String value check
        self.__result["not_in_value_list"] = self.not_in_value_list()
        return self.__result